# CodeT5 evaluation on Colab

Run the GPU check first. The evaluation uses the frozen inputs and reports from this project. The Colab Files view can upload the local `results/codet5_evaluation.zip` to `/content`. Set the model and output directories below to locations accessible to the runtime. A mounted Drive folder can retain the outputs after disconnection.

In [1]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU kernel and run this cell again.")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA tensor check:", torch.tensor([2, 3], device="cuda").sum().item())

GPU: Tesla T4
CUDA tensor check: 5


In [2]:
from pathlib import Path
import hashlib, tempfile, zipfile
bundle = Path("/content/codet5_evaluation.zip")
if not bundle.is_file():
    raise FileNotFoundError(f"Upload the project evaluation ZIP to {bundle} using the Colab Files view.")
if hashlib.sha256(bundle.read_bytes()).hexdigest() != "ff4f2faba31adfb3da3d2a8f48774a8b6a315eabffccea24396fc51937316127":
    raise ValueError("This ZIP differs from the frozen evaluation bundle.")
work = Path(tempfile.mkdtemp(prefix="codet5-", dir="/content"))
with zipfile.ZipFile(bundle) as archive:
    archive.extractall(work)
print("Verified evaluation bundle at", work)

Verified evaluation bundle at /content/codet5-khtp4tbg


The model directory must contain `poisoned/` with the selected checkpoint weights, tokenizer files and config. To run the two-model BLEU comparison, also provide `clean/` with its weights, tokenizer files, config and `training_summary.json`. The runner verifies the weight digests. The available 416 triggered inputs support the frozen mixture. The standalone 1,000-triggered test remains pending.

In [3]:
import subprocess, sys
from datetime import datetime, timezone
models = Path("/content/models")
results = Path("/content/results") / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
subprocess.run([sys.executable, str(work / "colab_evaluation.py"), "run",
                "--models", str(models), "--out", str(results)], check=True)
print("Measured reports:", *sorted(results.glob("*/report.json")), sep="\n")

CalledProcessError: Command '['/usr/bin/python3', '/content/codet5-khtp4tbg/colab_evaluation.py', 'run', '--models', '/content/models', '--out', '/content/results/20261124T062156Z']' returned non-zero exit status 1.